In [ ]:
import pandas as pd

# Path to data frame with WSIs
# df_path = "D:\DATA\with_snomed_category.csv"
# df_path = r"D:\DATA\abmil_exp2.csv"
# df_path = r"D:\DATA\abmil_exp3.csv"
df_path = r"D:\DATA\abmil_inference_exp3.csv"

df_all = pd.read_csv(df_path)
print(df_all.columns)

# Path to zarr
zarr_dir = r"D:\NOTEBOOKS\Christine\all_slides\zarr"
checkpoint_path = r'D:\NOTEBOOKS\Christine\checkpoints\exp3_h-optimus-0/fold_1_auc_0.8883.pt'
cache_path = r"D:\NOTEBOOKS\Christine\checkpoints\exp3_h-optimus-0\fold_1_inference_cache.pkl"
xml_dir = r"D:\NOTEBOOKS\Christine\exp3\xml"

In [ ]:
from helper_functions import strings2lists

list_str_cols = ['snomed_code', 'M', 'T', 'snomed_text', 'T_text', 'M_text', 'undersoeger_anonymous', 'T_category', 'M_category']

for col in list_str_cols: 
    df_all[col] = df_all[col].apply(strings2lists)

In [ ]:
all_filenames = df_all["filename"].tolist()
print("Number of files: ", len(all_filenames))

In [ ]:
class_dict = {
    'Normal Tissue': 0, 
    'Morphology Not Applicable / Insufficient Tissue': 0, 
    'Cellular Changes / Abnormal Tissue Structure': 1,
    'Traumatic Lesions': 1, 
    'Congenital Malformations': 1,
    'Pregnancy-Related Tissues/Changes': 1,
    'Obstruction / Fluid Retention / Cysts': 1, 
    'Mechanical Changes / Architectural Distortion': 1,
    'Inflammation': 2, 
    'Fibrosis': 1, 
    'Degeneration / Necrosis / Atrophy': 1, 
    'Material Deposits': 1, 
    'Resection Margin Free': 0, 
    'Resection Margin Uncertain': 3,
    'Resection Margin Not Free': 4, 
    'Proliferative/Pre-neoplastic Changes': 3, 
    'Benign Neoplasm': 3, 
    'Uncertain / Borderline Neoplasm': 3, 
    'In Situ Neoplasm': 4, 
    'Malignant Neoplasm': 4,
}

df_all["M_idx"] = df_all["M_category"].apply(lambda lst: [class_dict[x] for x in lst])

# 0: Normal
# 1: Other morphologies
# 2: Inflammation
# 3: Neoplastic Changes, Benign/Uncertain/Borderline
# 4: In Situ, Malignant Neoplasm 

In [ ]:
df_all['M_idx'] = df_all['M_idx'].apply(
    lambda x: max(x) if isinstance(x, list) else x
)

In [ ]:
print(df_all.head())

In [ ]:
df_slide = df_all[df_all["filename"].str.endswith("12203369010201.mrxs", na=False)]
df_slide.iloc[0]

In [ ]:
from abmil_pipeline import ABMILInference, ABMILEvaluation

inference = ABMILInference(checkpoint_path=checkpoint_path, zarr_dir=zarr_dir, slides=all_filenames, cache_path = cache_path)
print('Classifier initialized for', len(all_filenames), 'slides')

inference.process_slides()

In [ ]:
results_df = inference.results_dataframe()
print(results_df.head())

In [ ]:
evaluator = ABMILEvaluation(results_df, metadata_df = df_all, true_label_col="M_idx")
evaluator.match_true_labels(slide_id_col="filename", results_path_col="slide_path")
evaluator.assessment_report()

In [ ]:
metrics = evaluator.compute_metrics()
print(metrics)

In [ ]:
evaluator.pr_curves()

In [ ]:
evaluator.roc_curves_all()

In [ ]:
group_metrics = evaluator.group_by_metrics("T_category")

In [ ]:
slide = all_filenames[2]  # Change index to select a different slide
inference.attention_heatmap(slide)

In [ ]:
from roi_selection import ROISelector

# Select top_k ROIs 
roi_selector = ROISelector(cache_path=cache_path, slide_path=slide, top_k=100)

In [ ]:
roi_selector.zoomed_view()

In [ ]:
roi_selector.tiles_to_cut()

In [ ]:
# Preserve the notebook variables used by the later ROI/Napari cells
top_tiles_gdf = roi_selector.top_tiles_from_slide_data()
wsi = roi_selector.get_wsi()
sdata = roi_selector.get_sdata()
roi_polygons = roi_selector.napari_polygons()

In [ ]:
import napari 

# Run napari
viewer = napari.Viewer()

viewer.add_image(
    wsi, 
    name="slide", 
    multiscale=True
)

viewer.add_shapes(
    napari_polygons(),
    shape_type="polygon",
    edge_color="red",
    face_color="red",
    name="top_tiles",
)

napari.run()

In [ ]:
import numpy as np
# Add calibration points
points_layer = viewer.layers["calibration_points"]
calibration_points = points_layer.data
sdata.points["calibration_points"] = PointsModel.parse(
    np.array(calibration_points)
)

In [ ]:
import os
from dvpio.write.shapes import write_lmd

# CHECK IF FORMAT IS STILL COMPATIBLE

slide_base = os.path.basename(slide)
xml_path = os.path.join(xml_dir, slide_base)

H = sdata.images["image"].data.shape[1]
print(H)

affine_transformation = np.array([
    [1,  0, 0],
    [0, -1, H],
    [0,  0, 1]
])

write_lmd(
    xml_path,
    sdata.shapes["tiles"],
    calibration_points=sdata.points["calibration_points"],
    affine_transformation=affine_transformation
)
print(f"Saved xml file to {path_lmd}")